In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
import gradio as gr

from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration
)
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer

In [ ]:
print("Loading T5 Model...")

tokenizer = T5Tokenizer.from_pretrained(
    "t5-small"
)

model = T5ForConditionalGeneration.from_pretrained(
    "t5-small"
)

print("Model Loaded Successfully")

In [ ]:
def extractive_summary(text, sentence_count=3):

    parser = PlaintextParser.from_string(
        text,
        Tokenizer("english")
    )

    summarizer = LsaSummarizer()

    summary = summarizer(
        parser.document,
        sentence_count
    )

    result = ""

    for sentence in summary:
        result += str(sentence) + " "

    return result

In [ ]:
def abstractive_summary(text, length):

    input_text = "summarize: " + text

    inputs = tokenizer.encode(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    if length == "Short":
        max_len = 60
        min_len = 20

    elif length == "Medium":
        max_len = 120
        min_len = 50

    else:  # Long
        max_len = 220
        min_len = 100

    summary_ids = model.generate(
        inputs,
        max_length=max_len,
        min_length=min_len,
        num_beams=5,
        no_repeat_ngram_size=2,
        length_penalty=0.8,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
def word_count(text):
    return len(text.split())

In [ ]:
def compression_ratio(original, summary):

    original_words = len(original.split())

    summary_words = len(summary.split())

    ratio = (
        summary_words /
        original_words
    ) * 100

    return f"{ratio:.2f}%"

In [ ]:
def summarize(text, file, length):

    if file is not None:

        with open(file.name, "r", encoding="utf-8") as f:
            text = f.read()

    ext_summary = extractive_summary(text)

    abs_summary = abstractive_summary(
        text,
        length
    )

    original_words = word_count(text)

    summary_words = word_count(abs_summary)

    ratio = compression_ratio(
        text,
        abs_summary
    )

    return (
        original_words,
        summary_words,
        ratio,
        ext_summary,
        abs_summary
    )



In [ ]:
def run_summarization(input_method, text, file, length):

    if file is not None:

        if not file.name.endswith(".txt"):
            return "❌ Invalid file type. Please upload a .txt file only.", None

        with open(file.name, "r", encoding="utf-8") as f:
            text = f.read()

    if text is None or len(text.strip()) == 0:
        return "❌ Please enter some text.", None

    try:
        ext_summary = extractive_summary(text)
        abs_summary = abstractive_summary(text, length)

    except Exception as e:
        return f"❌ Error occurred: {str(e)}", None

    original_words = len(text.split())
    summary_words = len(abs_summary.split())
    ratio = f"{(summary_words / original_words) * 100:.2f}%"

    result = f"""
     📏 Summary Length Selected: {length}
📌 Original Words: {original_words}
📌 Summary Words: {summary_words}
📌 Compression: {ratio}

---

🔹 Extractive Summary:
{ext_summary}

---

🔸 Abstractive Summary:
{abs_summary}
"""

    file_path = "summary_output.txt"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(result)

    return result, file_path

In [ ]:
custom_css = """
.gradio-container {
    background: #0f172a !important;
    color: white !important;
    font-family: Arial;
}

.banner {
    background: linear-gradient(90deg, #1e293b, #2563eb);
    padding: 18px;
    border-radius: 12px;
    margin-bottom: 15px;
}

textarea, input, select {
    background-color: #111827 !important;
    color: white !important;
    border: 1px solid #334155 !important;
}

button {
    background: #2563eb !important;
    color: white !important;
    font-weight: bold !important;
}
"""

In [ ]:
with gr.Blocks(css=custom_css) as demo:

    # HEADER
    gr.Markdown("""
    <div class="banner">
        <h1>🧠 NLP Text Summariser</h1>
        <p>Extractive + Abstractive</p>
    </div>
    """)

    # INPUT MODE
    gr.Markdown("## 📥 Input Mode")

    input_method = gr.Radio(
        ["Paste Text", "Upload File"],
        value="Paste Text"
    )

    text_input = gr.Textbox(
        lines=8,
        placeholder="Paste your text here...",
        visible=True
    )

    file_input = gr.File(
    file_types=[".txt"],
    label="Upload TXT File Only",
    visible=False
)

    # toggle logic
    def toggle(choice):
        if choice == "Paste Text":
            return gr.update(visible=True), gr.update(visible=False)
        return gr.update(visible=False), gr.update(visible=True)

    input_method.change(toggle, input_method, [text_input, file_input])

    # SETTINGS
    gr.Markdown("## ⚙️ Settings")

    summary_length = gr.Radio(
        ["Short", "Medium", "Long"],
        value="Medium"
    )
    # BUTTON
    submit = gr.Button("🚀 Generate Summary")

    # OUTPUT
    gr.Markdown("## 📊 Output")

    output_text = gr.Textbox(lines=10, label="Summary")
    output_file = gr.File(
    label="📥 Download Summary"
)

    submit.click(
        fn=run_summarization,
        inputs=[input_method, text_input, file_input,
                summary_length],
        outputs=[output_text, output_file]
    )

demo.launch()